# 01 — Data processing

Gabriela's scope: audit the Roboflow YOLO export `garbage-classification-3`, clean annotations, and build our own train/val/test split (test stays held-out).

Raw data is **not modified**. Audit manifests go to `data/interim/`; the training-ready dataset goes to `data/processed/`.

CLI equivalent:

```bash
uv run python scripts/process_data.py
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
from pathlib import Path

from vpc2.data import io
from vpc2.data.processing import collect_yolo_files, run_pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"

dataset_root = io.discover_raw_dataset(RAW_DIR)
yaml_payload = io.load_yolo_yaml(dataset_root / "data.yaml")
class_names = io.class_names_from_yaml(yaml_payload)
images, labels = collect_yolo_files(dataset_root)

print(f"dataset_root: {dataset_root}")
print(f"classes ({len(class_names)}): {class_names}")
print(f"images: {len(images)}  labels: {len(labels)}")

## Pipeline: audit → clean → split → write

If the unzip is incomplete (for example missing `train/labels` or `valid/`), the report flags it and only uses valid image+label pairs.

In [ ]:
report = run_pipeline(
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    processed_dir=PROCESSED_DIR,
    seed=42,
    ratios=(0.70, 0.20, 0.10),
)

print(
    {
        "paired_kept": report.paired_kept,
        "corrupt": report.images_corrupt,
        "orphan_images": report.orphan_images,
        "orphan_labels": report.orphan_labels,
        "empty_labels": report.empty_labels,
        "boxes_clipped": report.boxes_clipped,
        "duplicates_removed": report.duplicates_removed,
        "split": {
            "train": report.split_train,
            "val": report.split_val,
            "test": report.split_test,
        },
        "class_histogram": report.class_histogram,
    }
)

## Handoff for the rest of the team

- Alejandro (augmentation): `data/processed/` — do **not** augment the test split.
- Julia (train): `data/processed/data.yaml`

In [ ]:
report_path = INTERIM_DIR / "audit" / "report.json"
yaml_path = PROCESSED_DIR / "data.yaml"

print("audit report:", report_path if report_path.exists() else "missing")
print("data.yaml:")
print(yaml_path.read_text(encoding="utf-8") if yaml_path.exists() else "missing")

if report.warnings:
    print("\nWarnings:")
    for warning in report.warnings:
        print("-", warning)